<a href="https://colab.research.google.com/github/EunjeLee0812/Sanhak/blob/seowonryeol/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# [환경 설정] 필수 라이브러리 설치
# faster-whisper: 음성 인식(STT) 모델
# rapidfuzz: 문자열 유사도 계산 (Levenshtein 거리 등) - 정답과 예측값 비교용
# g2pk: 한국어 발음 변환기 (Grapheme-to-Phoneme)
# konlpy, python-mecab-ko: 한국어 형태소 분석기 (단어 단위 오류율 측정용)
!pip install faster-whisper rapidfuzz g2pk konlpy python-mecab-ko

In [17]:
# 1. 시스템 및 유틸리티 라이브러리
import gc, sys
import os, re, json, glob, csv, random, glob, time
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
import importlib

# 2. 자연어 처리 및 문자열 매칭 라이브러리
from g2pk import G2p  # 발음 변환
from rapidfuzz.distance import Levenshtein # 편집 거리 계산
from rapidfuzz import process, fuzz # 문자열 유사도 처리
from mecab import MeCab # 형태소 분석

# 3. AI 모델 라이브러리
from faster_whisper import WhisperModel # Whisper ASR 모델

In [18]:
# # [Google Colab 전용] 구글 드라이브 마운트
# # 로컬 환경이 아닌 Colab에서 실행 시 주석을 해제하여 드라이브를 연결합니다.
# from google.colab import drive
# drive.mount('/content/drive')

# [개발 편의 설정] 모듈 자동 리로드
# 외부 .py 파일을 수정했을 때, 커널(세션)을 껐다 켜지 않아도
# 자동으로 변경 사항이 반영되도록 설정합니다.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
# [경로 설정]
# 프로젝트 파일들이 위치한 경로를 설정하거나 이동합니다. (현재는 주석 처리됨)
# BASE_PATH = "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/"
# %cd /content/drive/MyDrive/0208

# [사용자 정의 모듈 Import]
# config.settings: 실험에 필요한 상수 및 설정값(경로, 모델명 등) 로드
from config.settings import *

# utils: 텍스트 정규화, 데이터 로더, 성능 평가 지표(CER/WER) 계산 도구
from utils.normalizer import TextNormalizer
from utils.data_loader import load_transcripts
from utils.metrics import calculate_cer, calculate_wer, evaluate_proper_nouns

# core: ASR 엔진, 핫워드 편향(Bias) 관리, 후처리 로직
from core.asr_engine import ASR
from core.bias_manager import BiasManager
from core.post_processor import postprocess_with_hotwords
from utils.exporter import *

In [20]:
# [메모리 관리] GPU 메모리 누수를 방지하기 위한 캐시 초기화 (필요시 주석 해제)
# gc.collect()
# torch.cuda.empty_cache()

# ==============================================================================
# [실험 설정 및 초기화]
# ==============================================================================
import os
from config.settings import RESULTS_DIR
import random 

# 실험의 재현성을 위해 랜덤 시드 고정
random.seed(42)

# 1. 결과 저장 디렉토리 생성
os.makedirs(RESULTS_DIR, exist_ok=True)

# 2. 결과 파일명 설정 (실행 시점의 타임스탬프 포함하여 덮어쓰기 방지)
now = time.gmtime(time.time()+(9*3600)) # 한국 시간(KST) 기준
formatted = time.strftime("[%Y%m%d_%H%M]", now)
OUT_ROWS = f"./results/{formatted}_asr_detail.csv"   # 상세 결과 저장용
OUT_SUM  = f"./results/{formatted}_asr_summary.csv"  # 요약 결과 저장용

# 3. 핵심 객체 초기화
normalizer = TextNormalizer()   # 텍스트 정규화기
mecab = MeCab()                 # 형태소 분석기
bias_mgr = BiasManager(BIAS_PATH) # 핫워드 가중치(Bias) 관리자
transcripts = load_transcripts(TRANSCRIPTS_PATH) # 정답 스크립트 로드
files = glob.glob(os.path.join(AUDIO_FOLDER, "**/*.mp4"), recursive=True) # 오디오 파일 목록 로드

# 4. ASR 모델 로드 (Whisper)
# initial_prompt를 통해 한국어 문맥을 유도
asr = ASR(ASR_MODEL, ASR_DEVICE, ASR_COMPUTE, initial_prompt=KOREAN_ONLY_PROMPT)

rows: List[Dict[str, Any]] = []  # 실험 결과를 한 줄씩 저장할 리스트

# ==============================================================================
# [메인 실험 루프]
# 다양한 파라미터 조합(Grid Search)을 통해 최적의 설정을 탐색합니다.
# ==============================================================================

# Loop 1: 핫워드(Hotword) 개수 설정 (Top-K)
for top_k in HOTWORD_TOPK_SWEEP: 
    
    # Loop 2: 핫워드 추출 전략 (랜덤 vs 하이브리드 등)
    for hotwords_strategy in HOTWORD_STRATEGY_SWEEP: 

        # Loop 3: 편향(Bias) 가중치 업데이트 반복 횟수
        for bias_weight_update_cnt in BIAS_WEIGHT_UPDATE_ITERATION_SWEEP: 
            
            # (옵션) 새로운 전략 시작 시 기존 학습된 편향 초기화
            if RESET_BIASING_LIST:
                bias_mgr.reset_biasing_list(BIAS_PATH)
            
            # ✅ Feedback Loop: 설정된 횟수만큼 반복하며 핫워드 인식을 강화
            for repeat in range(bias_weight_update_cnt):

                # [Step 1] 현재 전략에 맞춰 핫워드 리스트 추출
                current_hotwords = bias_mgr.get_weighted_hotwords(top_k, mode=hotwords_strategy)
                
                # Loop 4: 후처리(Post-processing) 적용 여부
                for pp_on in POSTPROCESS_SWEEP:
                    pp_str= "ON" if pp_on ==1 else "OFF"
                    print(f"\n[RUN] Top-K: {top_k} | Iteration: {repeat+1}/{bias_weight_update_cnt} | PostProcess: {pp_str}")
                    print(f"hotwords : {current_hotwords}\n")

                    # 데이터 편향 방지를 위해 파일 처리 순서를 매번 섞음
                    random.shuffle(files) 

                    # [Step 2] 개별 오디오 파일 처리 (최대 AUDIO_FILE_MAX개 만큼)
                    for audio_path in files[:AUDIO_FILE_MAX]:
                        fname = os.path.basename(audio_path)
                        # 해당 파일의 정답(Ground Truth) 데이터 가져오기
                        meta = transcripts.get(fname, {"text": "", "entities": []})

                        # 1) ASR 추론: Whisper 모델로 음성을 텍스트로 변환 (핫워드 적용)
                        hyp_raw = asr.transcribe(audio_path, "ko", ASR_BEAM, hotwords=current_hotwords)
                        
                        # 2) 후처리: 핫워드 기반 오타 교정 (옵션)
                        if pp_on:
                            hyp_final, replog = postprocess_with_hotwords(
                                hyp_raw, current_hotwords, normalizer,
                                gate=RULE_GATE, tol=RULE_TOL, wratio_th=RULE_WRATIO_TH
                            )
                        else:
                            hyp_final, replog = normalizer.noramlize(hyp_raw), []
                        
                        # 정답 텍스트 정규화 (비교를 위해 포맷 통일)
                        ref_final=normalizer.normalize(meta["text"], False)

                        # ✅ 3) 고유명사(PN) 성능 평가
                        # Recall, CER, WER 등 다양한 지표를 계산하여 반환
                        pn_recall, wrong_pn_char_cnt, pn_char_cnt, pn_cer, wrong_pn_morph_cnt, pn_morph_cnt, pn_wer, hyp_ents, soft_missed_ents = evaluate_proper_nouns(
                            meta.get("entities", []), hyp_final, normalizer, match_th=PN_MATCH_TH)
                        
                        # 4) 전체 문장 성능 평가 (CER: 음절 오류율, WER: 단어 오류율)
                        cer, wrong_char_cnt, char_cnt = calculate_cer(ref_final, hyp_final, normalizer)
                        wer, wrong_morph_cnt, morph_cnt, ref_morphs, hyp_morphs = calculate_wer(ref_final, hyp_final, normalizer, mecab,meta["entities"], hyp_ents)

                        # ✅ [중요] 피드백 반영: 인식에 실패한 고유명사(Hard Miss)를 학습하여 다음 루프에 반영
                        bias_mgr.add_miss(soft_missed_ents)

                        # 결과 출력을 위한 포맷팅
                        pn_recall_disp = f"{pn_recall:.4f}" if pn_recall is not None else "NA"
                        pn_cer_disp    = f"{pn_cer:.4f}"    if pn_cer    is not None else "NA"

                        # [로그 출력] 콘솔에 현재 파일의 처리 결과 표시
                        print(
                            f"- file: {os.path.dirname(audio_path).split('/')[-1]}/{fname} | "
                            f"pp_on={pp_on} | cer={cer:.4f} | wer={wer:.4f} | pn_cer={pn_cer_disp} | pn_wer={pn_wer:.4f} | pn_recall={pn_recall_disp}" 
                        )
                        print(
                            f"ref_text:  [{meta['text']}]\n"
                            f"hyp_raw:   [{hyp_raw}]\n"
                            f"hyp_final: [{hyp_final}]\n"
                            f"ref_pn:    {meta.get('entities', [])}\n"
                            f"hyp_pn:    {hyp_ents}\n"
                            f"soft_miss: {soft_missed_ents}\n"
                        )

                        # [결과 수집] CSV 저장을 위해 딕셔너리 형태로 데이터 저장
                        rows.append({
                            "file": f"{os.path.dirname(audio_path).split('/')[-1]}/{fname}",
                            "top_k": top_k,
                            "postprocess_on": int(pp_on),
                            "hotwords_strategy": "random" if hotwords_strategy == 1 else "hybrid",
                            "bias_weight_update_cnt": repeat+1, 
                            "hotwords": current_hotwords,

                            # 주요 성능 지표
                            "cer": f"{cer:.4f}",
                            "wer": f"{wer:.4f}",
                            "pn_cer": None if pn_cer is None else float(f"{pn_cer:.4f}"),
                            "pn_wer": None if pn_wer is None else float(f"{pn_wer:.4f}"),
                            "pn_recall": None if pn_recall is None else float(f"{pn_recall:.4f}"),

                            # 텍스트 데이터 (정답 vs 예측)
                            "ref_raw": meta["text"],
                            "hyp_raw": hyp_raw,
                            "ref_final" : ref_final,
                            "hyp_final": hyp_final,

                            # 고유명사 분석 데이터
                            "ref_pn": meta.get("entities", []),
                            "hyp_pn": hyp_ents,
                            "soft_missed_pn": soft_missed_ents,

                            # 상세 분석용 데이터
                            "replog": json.dumps(replog, ensure_ascii=False),
                            "wrong_char_cnt":wrong_char_cnt,
                            "char_cnt":char_cnt,
                            "wrong_morph_cnt":wrong_morph_cnt,
                            "morph_cnt":morph_cnt,
                            "wrong_pn_char_cnt": wrong_pn_char_cnt, 
                            "pn_char_cnt": pn_char_cnt,
                            "wrong_pn_morph_cnt": wrong_pn_morph_cnt, 
                            "pn_morph_cnt": pn_morph_cnt,
                            "ref_morphs": ref_morphs,
                            "hyp_morphs":hyp_morphs
                        })

                # [Step 3] 가중치 업데이트: 한 번의 루프(repeat)가 끝나면 미인식된 단어들의 가중치를 갱신
                bias_mgr.finalize(repeat)

# 5. 실험 종료 후 결과 파일 저장
save_results_with_summary(rows, OUT_ROWS)

# (옵션) 요약본 별도 저장 코드 (현재 주석 처리됨)
# summary = summarize(rows)
# with open(OUT_SUM, "w", newline="", encoding="utf-8-sig") as f:
#     w = csv.DictWriter(f, fieldnames=list(summary[0].keys()))
#     w.writeheader()
#     w.writerows(summary)

print("\n[DONE] All experiments finished.")

[SUCCESS] /teamspace/studios/this_studio/Sanhak/lists/biasing_list.json 데이터가 모두 0으로 초기화되었습니다.

[RUN] Top-K: 20 | Iteration: 1/2 | PostProcess: ON
hotwords : ['가요톱텐', '강호동', '거침없이 하이킥', '겨울왕국 이', '겨울왕국', '결승전', '고잉 세븐틴', '골든 아워 더 비기닝', '요리', '그날 이야기', '파친코', '모래알갱이', '인턴기자', '유튜브', '르세라핌', '주기자', '내역', '나영석', '리마스터링', '시그널']

- file: New_Audio_Files/269.mp4 | pp_on=1 | cer=0.0000 | wer=0.0000 | pn_cer=0.0000 | pn_wer=0.0000 | pn_recall=1.0000
ref_text:  [에스비에스 드라마 재방송 알림 십 분 전으로 설정해줘]
hyp_raw:   [sbs 드라마 재방송 알림 10분 전으로 설정해줘]
hyp_final: [에스비에스 드라마 재방송 알림 십분 전으로 설정해줘]
ref_pn:    ['에스비에스']
hyp_pn:    ['에스비에스']
soft_miss: []

- file: New_Audio_Files/140.mp4 | pp_on=1 | cer=0.0000 | wer=0.0000 | pn_cer=0.0000 | pn_wer=0.0000 | pn_recall=1.0000
ref_text:  [이비에스에서 대학생 대상 교양 강의 영상 찾아줘]
hyp_raw:   [EBS에서 대학생대상교양강의영상찾아줘]
hyp_final: [이비에스에서 대학생대상교양강의영상찾아줘]
ref_pn:    ['이비에스']
hyp_pn:    ['이비에스']
soft_miss: []

- file: New_Audio_Files/081.mp4 | pp_on=1 | cer=0.0667 | wer=0.1176 | pn_cer=0.0000 | pn